# 042 — Ingeniería y selección de características

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Transformaciones numéricas:** estandarizar `(x−μ)/σ` para modelos de distancia o
regularizados; log para colas largas; binning para no linealidades en lineales. Árboles:
invariantes a transformaciones monótonas.

**Categóricas:** one-hot (seguro, explota en cardinalidad), ordinal (solo con orden real),
target encoding (potente y peligroso: out-of-fold + suavizado
`(n·ȳ_c + k·ȳ)/(n+k)` obligatorios), hashing (colisiones controladas).

**Selección:** filtro (correlación/MI, barato, ciego a interacciones) → embedded
(lasso, importancias) → wrapper (RFE con CV anidada, caro).

**Regla anti-fuga:** todo parámetro aprendido (μ, σ, encodings, vocabularios, subconjunto
seleccionado) se ajusta SOLO con train, dentro del pipeline que se valida. Seleccionar
features con el dataset completo y "luego validar" infla la métrica (ESL §7.10.2).


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** X: (200·0.15 + 20·0.08)/220 = (30 + 1.6)/220 ≈ **0.144**.
Y: (15·0.40 + 1.6)/35 = 7.6/35 ≈ **0.217**. Z: (1·1.00 + 1.6)/21 ≈ **0.124**.
Z se aleja más (de 1.00 a 0.124): con n=1 la "media observada" es una fila, no evidencia;
el suavizado impone que las categorías raras digan poco más que la media global —
exactamente el anti-sobreajuste deseado.

**Ejercicio 2.** Con 5 000 features aleatorias y n=100, por puro azar decenas tendrán
correlación alta con el target (la máxima de 5 000 correlaciones nulas es grande). Al
seleccionarlas mirando TODO el dataset, esas correlaciones espurias existen también en las
filas que luego serán "validación": la CV evalúa un pipeline que ya memorizó ruido. Con
selección dentro de cada fold, las features elegidas con el train del fold no tienen razón
para correlacionar en su validación: esperarías accuracy ≈ 0.5. Es el experimento de
ESL §7.10.2.

**Ejercicio 3.** h=23: (sin(2π·23/24), cos(2π·23/24)) ≈ (−0.259, 0.966).
h=1: (0.259, 0.966). Distancia = √((0.518)² + 0²) ≈ **0.518** — vecinos en el círculo,
como corresponde a las 23 h y la 1 h. En crudo distan 22 de 23 posibles: casi opuestos.
Se benefician los modelos de distancia y los lineales; a los árboles les da casi igual
(pueden partir la hora cruda en dos rangos, aunque necesitan dos splits para "cerrar" el
círculo).

**Ejercicio 4.** `log(x)` es monótona creciente: preserva el orden, así que existe un
umbral equivalente (t' = log t) con exactamente la misma accuracy — no cambia nada.
`−x` es monótona decreciente: invierte el orden; con umbrales de la forma `x ≥ t` basta
usar `−x ≥ −t` con la desigualdad invertida, así que la accuracy alcanzable tampoco
cambia, solo la dirección de la regla. La familia de clasificadores por umbral es
invariante a transformaciones monótonas de la feature.


In [ ]:
result = run_lab("ml", seed=42)
assert result["kind"] == "ml"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — target encoding suavizado
y_global, k = 0.08, 20

def encoding(n_c, y_c):
    return (n_c * y_c + k * y_global) / (n_c + k)

for nombre, n_c, y_c in [("X", 200, 0.15), ("Y", 15, 0.40), ("Z", 1, 1.00)]:
    print(f"ciudad {nombre}: media observada {y_c:.2f} → encoding {encoding(n_c, y_c):.3f}")
# La categoría con n=1 queda casi en la media global: no memoriza su única fila.


In [ ]:
# Ejercicio 3 — codificación cíclica
import math

def cic(h):
    ang = 2 * math.pi * h / 24
    return (math.sin(ang), math.cos(ang))

a, b = cic(23), cic(1)
d = math.dist(a, b)
print(f"h=23 → ({a[0]:.3f}, {a[1]:.3f});  h=1 → ({b[0]:.3f}, {b[1]:.3f})")
print(f"distancia cíclica = {d:.3f}   vs distancia cruda = 22")
# En el círculo, 23h y 1h son vecinas: la codificación respeta la geometría del tiempo.


## Reflexión

1. El laboratorio trabaja con una única feature sintética. Si le añadieras 99 features de
   ruido puro y seleccionaras "la mejor" por accuracy antes de validar, ¿qué accuracy de
   validación esperarías para la ganadora y por qué está inflada?
2. ¿Por qué el target encoding sin out-of-fold puede dar validación excelente y producción
   desastrosa, mientras el one-hot no tiene ese modo de fallo?
3. ¿Qué transformación de esta clase cambiaría el resultado del laboratorio (que elige un
   umbral sobre una feature) y cuál lo dejaría exactamente igual? Justifica con la
   invariancia monótona.
